# 🇲🇰 Vezilka v2 — Colab Pipeline Runner

Runs the full MK↔SQ parallel corpus pipeline on Colab's free T4 GPU.

**Prerequisites:**
- Your PDFs uploaded to a Google Cloud Storage bucket
- Upload structure: `gs://YOUR_BUCKET/pdfs/2001/01/xxx.pdf` (same year/month layout)

**Estimated runtime:** ~1.5–2 hours on T4 GPU (full 4,931 PDFs with validation)

## Step 0 — Check GPU & Set Your Bucket Name

Go to **Runtime → Change runtime type → T4 GPU** before running.

In [1]:
# === CONFIGURE THIS ===
GCS_BUCKET = "vezilka-pdfs"  # <-- Change to your bucket name
GCS_PDF_PREFIX = "pdfs/"     # <-- Prefix inside the bucket (pdfs/2001/01/...)
GCS_OUTPUT_PREFIX = "output/" # <-- Where to upload results
# ======================

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

import psutil
ram = psutil.virtual_memory().total / 1e9
print(f"RAM: {ram:.1f} GB")

import shutil
disk = shutil.disk_usage('/')
print(f"Disk: {disk.free / 1e9:.1f} GB free")

AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## Step 1 — Install Dependencies

In [ ]:
%%capture install_output
!pip install -q \
    pdfplumber pymupdf \
    lingua-language-detector langdetect \
    sacremoses sentence-splitter \
    transformers torch sacrebleu \
    sentence-transformers \
    laser_encoders \
    unbabel-comet \
    faiss-cpu \
    datasketch \
    pandas datasets numpy \
    tqdm matplotlib \
    google-cloud-storage

print("✅ All packages installed")

## Step 2 — Authenticate & Download PDFs from GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated with Google Cloud")

In [ ]:
import os
from google.cloud import storage
from pathlib import Path
from tqdm.auto import tqdm

PDF_LOCAL_DIR = Path("/content/vezilka_v2/data/pdfs")
PDF_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

# List all PDFs in the bucket
blobs = list(bucket.list_blobs(prefix=GCS_PDF_PREFIX))
pdf_blobs = [b for b in blobs if b.name.endswith(".pdf")]
print(f"Found {len(pdf_blobs)} PDFs in gs://{GCS_BUCKET}/{GCS_PDF_PREFIX}")

# Download with progress bar
downloaded = 0
skipped = 0
for blob in tqdm(pdf_blobs, desc="Downloading PDFs"):
    # Preserve directory structure: pdfs/2021/01/abc.pdf → /content/vezilka_v2/data/pdfs/2021/01/abc.pdf
    relative = blob.name[len(GCS_PDF_PREFIX):]  # strip the prefix
    local_path = PDF_LOCAL_DIR / relative
    if local_path.exists():
        skipped += 1
        continue
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob.download_to_filename(str(local_path))
    downloaded += 1

total_pdfs = len(list(PDF_LOCAL_DIR.rglob("*.pdf")))
print(f"\n✅ {downloaded} downloaded, {skipped} already existed, {total_pdfs} total PDFs on disk")

## Step 3 — Write Pipeline Code to Disk

All 21 modules written to `/content/vezilka_v2/`

In [ ]:
import os

BASE = "/content/vezilka_v2"
os.makedirs(f"{BASE}/utils", exist_ok=True)
os.makedirs(f"{BASE}/phase3_extract", exist_ok=True)
os.makedirs(f"{BASE}/phase4_align", exist_ok=True)
os.makedirs(f"{BASE}/phase5_clean", exist_ok=True)
os.makedirs(f"{BASE}/data/extracted", exist_ok=True)
os.makedirs(f"{BASE}/data/segmented", exist_ok=True)
os.makedirs(f"{BASE}/data/aligned", exist_ok=True)
os.makedirs(f"{BASE}/data/output", exist_ok=True)
print("✅ Directory structure created")

In [ ]:
%%writefile /content/vezilka_v2/config.py
"""
Vezilka v2 — Configuration Module (Colab version).
All tuneable parameters in one place.
"""
from __future__ import annotations
import os
from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class VezilkaConfig:
    project_root: Path = Path("/content/vezilka_v2")
    pdf_dir: Path = Path("/content/vezilka_v2/data/pdfs")
    data_dir: Path = Path("/content/vezilka_v2/data")
    extracted_dir: Path = Path("/content/vezilka_v2/data/extracted")
    segmented_dir: Path = Path("/content/vezilka_v2/data/segmented")
    aligned_dir: Path = Path("/content/vezilka_v2/data/aligned")
    output_dir: Path = Path("/content/vezilka_v2/data/output")
    failed_log: Path = Path("/content/vezilka_v2/data/failed_pdfs.jsonl")
    skipped_log: Path = Path("/content/vezilka_v2/data/skipped_pdfs.jsonl")
    rejected_pairs_log: Path = Path("/content/vezilka_v2/data/rejected_pairs.jsonl")

    # Gatekeeper
    gatekeeper_min_file_size_kb: int = 50
    gatekeeper_albanian_chars: frozenset = frozenset("ëËçÇ")
    gatekeeper_albanian_keywords: list = field(default_factory=lambda: [
        "Neni", "LIGJ", "ligj", "Maqedonisë", "Veriut", "denarë", "Gazeta",
    ])
    gatekeeper_min_latin_ratio_for_albanian: float = 0.20

    # Document Segmenter
    item_boundary_pattern: str = r"^\s*(\d{3,4})\.\s*$"
    segmenter_min_laser_doc_sim: float = 0.80

    # Layout Classification
    two_column_min_page_fraction: float = 0.50
    column_split_tolerance: float = 0.08
    cyrillic_threshold: float = 0.60
    latin_threshold: float = 0.60
    sequential_transition_threshold: float = 0.70
    albanian_boundary_patterns: list = field(default_factory=lambda: [
        r"^L\s*I\s*G\s*J\b", r"^LIGJ\b", r"^Neni\s+1\b", r"_{5,}", r"—{5,}",
    ])

    # Alignment
    structural_article_pattern_mk: str = r"Член\s+(\d+)"
    structural_article_pattern_sq: str = r"Neni\s+(\d+)"
    gc_mean_char_ratio: float = 1.1
    gc_variance: float = 6.8
    min_article_text_length: int = 10

    # Dense Retrieval
    dense_retrieval_min_similarity: float = 0.70
    dense_retrieval_use_faiss: bool = True
    dense_retrieval_monotonicity: bool = True
    dense_retrieval_min_pairs_fallback: int = 3
    gc_min_sentences_threshold: int = 10

    # Translation Scorer
    translation_model: str = "helsinki"
    translation_batch_size: int = 32
    run_translation_scoring: bool = True

    # LaBSE
    labse_model: str = "sentence-transformers/LaBSE"
    labse_batch_size: int = 256
    labse_min_similarity: float = 0.55

    # LASER3
    run_laser3: bool = True
    laser3_min_similarity: float = 0.55

    # COMET-QE
    run_comet_qe: bool = True
    comet_qe_model: str = "Unbabel/wmt22-cometkiwi-da"
    comet_qe_min_score: float = 0.65

    # Back-Translation
    run_back_translation: bool = True
    bt_mk_to_sq_model: str = "Helsinki-NLP/opus-mt-mk-sq"
    bt_sq_to_mk_model: str = "Helsinki-NLP/opus-mt-sq-mk"
    bt_batch_size: int = 32

    # Blended Weights — structural
    w_structural_labse: float = 0.50
    w_structural_laser3: float = 0.30
    w_structural_length: float = 0.20
    # Blended Weights — non-structural
    w_nonstructural_labse: float = 0.25
    w_nonstructural_laser3: float = 0.20
    w_nonstructural_comet_qe: float = 0.25
    w_nonstructural_backtranslation: float = 0.20
    w_nonstructural_length: float = 0.10
    blended_min_score: float = 0.70

    # Hard Rejection
    hard_reject_labse: float = 0.55
    hard_reject_comet_qe: float = 0.65
    hard_reject_laser3: float = 0.55
    hard_reject_length_ratio: float = 0.35

    # Filtering
    min_words: int = 5
    max_words: int = 200
    min_length_ratio: float = 0.4
    max_length_ratio: float = 2.5
    max_digit_fraction: float = 0.30
    mk_min_cyrillic: float = 0.50
    sq_min_latin: float = 0.50

    # Deduplication
    minhash_threshold: float = 0.95
    minhash_num_perm: int = 128

    # Export
    train_fraction: float = 0.80
    val_fraction: float = 0.10
    test_fraction: float = 0.10
    export_formats: list = field(default_factory=lambda: ["tsv", "jsonl"])

    # Logging
    log_level: str = "INFO"

    # GPU
    device: str = "cuda"

    def ensure_dirs(self) -> None:
        for d in (self.data_dir, self.extracted_dir, self.segmented_dir,
                  self.aligned_dir, self.output_dir):
            d.mkdir(parents=True, exist_ok=True)

DEFAULT_CONFIG = VezilkaConfig()

In [ ]:
%%writefile /content/vezilka_v2/utils/__init__.py
"""Utilities."""

In [ ]:
%%writefile /content/vezilka_v2/utils/logging_config.py
"""Logging setup."""
from __future__ import annotations
import logging
import sys
from pathlib import Path

def setup_logging(level: str = "INFO", log_file: str | None = None) -> None:
    root = logging.getLogger()
    root.setLevel(getattr(logging, level.upper(), logging.INFO))
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                            datefmt="%H:%M:%S")
    if not root.handlers:
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        root.addHandler(sh)
    if log_file:
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setFormatter(fmt)
        root.addHandler(fh)
    for name in ("urllib3", "filelock", "transformers.modeling_utils",
                 "sentence_transformers", "PIL", "matplotlib"):
        logging.getLogger(name).setLevel(logging.WARNING)

In [ ]:
%%writefile /content/vezilka_v2/utils/text_utils.py
"""Shared text processing functions."""
from __future__ import annotations
import re, unicodedata

HYPHEN_PATTERN = re.compile(r"(\w)\s*[-–]\s*([a-zа-ш])")
HEX_HASH_PATTERN = re.compile(r"\b[0-9a-f]{32,}\b", re.IGNORECASE)
MULTI_SPACE = re.compile(r"\s+")
PAGE_NUMBER = re.compile(r"^\s*\d{1,4}\s*$")
HEADER_FOOTER = re.compile(
    r"(СЛУЖБЕН\s*ВЕСНИК|GAZETTE\s*ZYRTARE|Стр\.\s*\d+\s*-\s*Бр\.\s*\d+|Faqe\s*\d+)",
    re.IGNORECASE)
DATE_HEADER = re.compile(
    r"\d{1,2}\s*(јануари|февруари|март|април|мај|јуни|јули|август|септември|"
    r"октомври|ноември|декември|janar|shkurt|mars|prill|maj|qershor|"
    r"korrik|gusht|shtator|tetor|nëntor|dhjetor)\s*\d{4}", re.IGNORECASE)
ALBANIAN_SPECIFIC_CHARS = frozenset("ëËçÇ")

def fix_albanian_encoding(text: str) -> str:
    return text.replace("`", "ë").replace("\x60", "ë")

def fix_hyphenation(text: str) -> str:
    return HYPHEN_PATTERN.sub(r"\1\2", text)

def remove_hex_hashes(text: str) -> str:
    return HEX_HASH_PATTERN.sub("", text).strip()

def normalise_whitespace(text: str) -> str:
    return MULTI_SPACE.sub(" ", text).strip()

def normalise_unicode(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def clean_text(text: str, is_albanian: bool = False) -> str:
    if not text: return ""
    text = normalise_unicode(text)
    if is_albanian: text = fix_albanian_encoding(text)
    text = fix_hyphenation(text)
    text = remove_hex_hashes(text)
    text = normalise_whitespace(text)
    return text

def cyrillic_ratio(text: str) -> float:
    alpha = [c for c in text if c.isalpha()]
    if not alpha: return 0.0
    return sum(1 for c in alpha if "\u0400" <= c <= "\u04FF") / len(alpha)

def latin_ratio(text: str) -> float:
    alpha = [c for c in text if c.isalpha()]
    if not alpha: return 0.0
    return sum(1 for c in alpha if c.isascii()) / len(alpha)

def has_albanian_markers(text: str) -> bool:
    return bool(ALBANIAN_SPECIFIC_CHARS.intersection(text))

def is_noise_line(line: str) -> bool:
    line = line.strip()
    if not line: return True
    if PAGE_NUMBER.match(line): return True
    if HEADER_FOOTER.search(line): return True
    if DATE_HEADER.search(line): return True
    return False

def strip_headers_footers(text: str) -> str:
    lines = text.split("\n")
    return "\n".join(l for l in lines if not is_noise_line(l))

def word_count(text: str) -> int:
    return len(text.split())

def digit_fraction(text: str) -> float:
    if not text: return 0.0
    return sum(c.isdigit() for c in text) / len(text)

def number_word_fraction(text: str) -> float:
    words = text.split()
    if not words: return 1.0
    return sum(1 for w in words if re.fullmatch(r"[\d.,/%]+", w)) / len(words)

In [ ]:
%%writefile /content/vezilka_v2/phase3_extract/__init__.py
"""Phase 3 — PDF text extraction."""

In [ ]:
%%writefile /content/vezilka_v2/phase3_extract/language_detector.py
"""Three-layer language detection: script → lingua → langdetect."""
from __future__ import annotations
import logging
from enum import Enum
logger = logging.getLogger(__name__)

class DetectedLanguage(Enum):
    MACEDONIAN = "mk"
    ALBANIAN = "sq"
    SERBIAN = "sr"
    UNKNOWN = "unknown"

class LanguageDetector:
    def __init__(self, cyrillic_threshold=0.60, latin_threshold=0.60):
        self.cyrillic_threshold = cyrillic_threshold
        self.latin_threshold = latin_threshold
        self._lingua = None

    def _get_lingua(self):
        if self._lingua is None:
            try:
                from lingua import Language, LanguageDetectorBuilder
                self._lingua = (
                    LanguageDetectorBuilder
                    .from_languages(Language.MACEDONIAN, Language.ALBANIAN, Language.SERBIAN)
                    .with_minimum_relative_distance(0.25)
                    .build())
            except ImportError:
                self._lingua = False
        return self._lingua if self._lingua is not False else None

    def detect(self, text: str) -> DetectedLanguage:
        if not text or len(text.strip()) < 3: return DetectedLanguage.UNKNOWN
        alpha = [c for c in text if c.isalpha()]
        if len(alpha) >= 5:
            cyr = sum(1 for c in alpha if "\u0400" <= c <= "\u04FF") / len(alpha)
            if cyr >= self.cyrillic_threshold: return DetectedLanguage.MACEDONIAN
            lat = sum(1 for c in alpha if c.isascii()) / len(alpha)
            if lat >= self.latin_threshold: return DetectedLanguage.ALBANIAN
        det = self._get_lingua()
        if det:
            try:
                from lingua import Language
                r = det.detect_language_of(text)
                if r == Language.MACEDONIAN: return DetectedLanguage.MACEDONIAN
                if r == Language.ALBANIAN: return DetectedLanguage.ALBANIAN
                if r == Language.SERBIAN: return DetectedLanguage.SERBIAN
            except: pass
        try:
            from langdetect import detect
            code = detect(text)
            m = {"mk": DetectedLanguage.MACEDONIAN, "sq": DetectedLanguage.ALBANIAN, "sr": DetectedLanguage.SERBIAN}
            if code in m: return m[code]
        except: pass
        return DetectedLanguage.UNKNOWN

In [ ]:
%%writefile /content/vezilka_v2/phase3_extract/gatekeeper.py
"""Pre-filter: decides if a PDF is worth processing."""
from __future__ import annotations
import json, logging
from dataclasses import dataclass
from enum import Enum
from pathlib import Path
import pdfplumber
from config import DEFAULT_CONFIG, VezilkaConfig
from utils.text_utils import latin_ratio
logger = logging.getLogger(__name__)

class SkipReason(Enum):
    NONE = "none"; FILE_TOO_SMALL = "file_too_small"
    NO_ALBANIAN_DETECTED = "no_albanian_detected"
    SCANNED_NO_TEXT_LAYER = "scanned_no_text_layer"
    CORRUPT_OR_UNREADABLE = "corrupt_or_unreadable"

@dataclass
class ProcessingDecision:
    should_process: bool; reason: SkipReason; detail: str = ""; is_scanned: bool = False

class Gatekeeper:
    def __init__(self, config=None): self.cfg = config or DEFAULT_CONFIG

    def check(self, pdf_path: Path) -> ProcessingDecision:
        try: size_kb = pdf_path.stat().st_size / 1024
        except OSError as e: return self._skip(SkipReason.CORRUPT_OR_UNREADABLE, str(e))
        if size_kb < self.cfg.gatekeeper_min_file_size_kb:
            return self._skip(SkipReason.FILE_TOO_SMALL, f"{size_kb:.0f} KB")
        try:
            with pdfplumber.open(str(pdf_path)) as pdf:
                total = len(pdf.pages)
                if total == 0: return self._skip(SkipReason.CORRUPT_OR_UNREADABLE, "0 pages")
                total_chars = 0
                sample = self._sample(total)
                texts = {}
                for i in sample:
                    t = pdf.pages[i].extract_text() or ""
                    texts[i] = t; total_chars += len(t.strip())
                if total_chars < 20:
                    for i in range(total):
                        if i not in texts:
                            total_chars += len((pdf.pages[i].extract_text() or "").strip())
                            if total_chars >= 20: break
                    if total_chars < 20:
                        return self._skip(SkipReason.SCANNED_NO_TEXT_LAYER, f"{total_chars} chars")
                if any(self._has_alb(t) for t in texts.values()):
                    return ProcessingDecision(True, SkipReason.NONE)
                for i in range(total):
                    if i in texts: continue
                    if self._has_alb(pdf.pages[i].extract_text() or ""):
                        return ProcessingDecision(True, SkipReason.NONE)
                return self._skip(SkipReason.NO_ALBANIAN_DETECTED, f"{total} pages")
        except Exception as e:
            return self._skip(SkipReason.CORRUPT_OR_UNREADABLE, str(e))

    def log_skip(self, pdf_path, decision):
        p = self.cfg.skipped_log; p.parent.mkdir(parents=True, exist_ok=True)
        try:
            with open(p, "a") as f:
                f.write(json.dumps({"pdf": str(pdf_path), "reason": decision.reason.value, "detail": decision.detail}, ensure_ascii=False) + "\n")
        except: pass

    @staticmethod
    def _sample(n):
        if n == 1: return [0]
        if n == 2: return [0, 1]
        return [0, n // 2, n - 1]

    def _has_alb(self, text):
        if not text: return False
        if any(c in text for c in self.cfg.gatekeeper_albanian_chars): return True
        for kw in self.cfg.gatekeeper_albanian_keywords:
            if kw in text: return True
        if latin_ratio(text) > self.cfg.gatekeeper_min_latin_ratio_for_albanian: return True
        return False

    @staticmethod
    def _skip(reason, detail=""):
        return ProcessingDecision(False, reason, detail, reason == SkipReason.SCANNED_NO_TEXT_LAYER)

In [ ]:
%%writefile /content/vezilka_v2/phase3_extract/layout_classifier.py
"""Classify PDF layout: TWO_COLUMN / SEQUENTIAL / MIXED."""
from __future__ import annotations
import logging, re
from dataclasses import dataclass
from enum import Enum
from pathlib import Path
from typing import Optional
import pdfplumber
from utils.text_utils import cyrillic_ratio, latin_ratio
logger = logging.getLogger(__name__)

class LayoutType(Enum):
    TWO_COLUMN = "two_column"; SEQUENTIAL = "sequential"
    MIXED_PRE2019 = "mixed"; SINGLE_LANGUAGE = "single"; UNKNOWN = "unknown"

@dataclass
class LayoutClassificationResult:
    layout_type: LayoutType; confidence: float; has_albanian: bool
    boundary_page: Optional[int] = None; boundary_block: Optional[int] = None
    total_pages: int = 0; detail: str = ""

ALB_BOUNDARY = [re.compile(r"^L\s*I\s*G\s*J\b", re.MULTILINE), re.compile(r"^LIGJ\b", re.MULTILINE), re.compile(r"^Neni\s+1\b", re.MULTILINE)]
SEP = re.compile(r"_{5,}|—{5,}")

class LayoutClassifier:
    def __init__(self, column_split_tolerance=0.08, two_column_min_fraction=0.50,
                 cyrillic_threshold=0.60, latin_threshold=0.60, sequential_transition_threshold=0.70):
        self.tol = column_split_tolerance; self.min_frac = two_column_min_fraction
        self.cyr_t = cyrillic_threshold; self.lat_t = latin_threshold; self.seq_t = sequential_transition_threshold

    def classify(self, pdf_path: Path) -> LayoutClassificationResult:
        try:
            with pdfplumber.open(str(pdf_path)) as pdf:
                total = len(pdf.pages)
                if total == 0: return LayoutClassificationResult(LayoutType.UNKNOWN, 0, False, detail="Empty")
                pd_list = self._analyse(pdf)
                has_alb = any(p["has_latin"] for p in pd_list)
                if not has_alb:
                    return LayoutClassificationResult(LayoutType.SINGLE_LANGUAGE, 0.9, False, total_pages=total)
                r = self._check_two_col(pd_list, total)
                if r: return r
                r = self._check_seq(pd_list, total)
                if r: return r
                return LayoutClassificationResult(LayoutType.MIXED_PRE2019, 0.5, has_alb, total_pages=total)
        except Exception as e:
            return LayoutClassificationResult(LayoutType.UNKNOWN, 0, False, detail=str(e))

    def _analyse(self, pdf):
        data = []
        for i, page in enumerate(pdf.pages):
            words = page.extract_words(use_text_flow=True) or []
            full = " ".join(w.get("text", "") for w in words)
            cr, lr = cyrillic_ratio(full), latin_ratio(full)
            w = float(page.width) if page.width else 612.0
            mid, t = w / 2, w * self.tol
            lw = [x for x in words if float(x.get("x0", 0)) < mid - t]
            rw = [x for x in words if float(x.get("x0", 0)) >= mid + t]
            lt = " ".join(x.get("text", "") for x in lw)
            rt = " ".join(x.get("text", "") for x in rw)
            bi = len(lw) > 10 and len(rw) > 10 and cyrillic_ratio(lt) > self.cyr_t and latin_ratio(rt) > self.lat_t
            data.append({"page_index": i, "full_text": full, "cyrillic_ratio": cr, "latin_ratio": lr,
                         "has_latin": lr > 0.20, "has_cyrillic": cr > 0.20, "is_bilingual_columns": bi})
        return data

    def _check_two_col(self, pd_list, total):
        bi = sum(1 for p in pd_list if p["is_bilingual_columns"])
        f = bi / max(total, 1)
        if f >= self.min_frac:
            return LayoutClassificationResult(LayoutType.TWO_COLUMN, min(f+0.1, 1), True, total_pages=total,
                                              detail=f"{bi}/{total} bilingual")

    def _check_seq(self, pd_list, total):
        if total < 2: return None
        boundary = self._find_marker(pd_list) or self._find_transition(pd_list)
        if boundary is None: return None
        before, after = pd_list[:boundary], pd_list[boundary:]
        mc = sum(1 for p in before if p["cyrillic_ratio"] > 0.5) / max(len(before), 1)
        sl = sum(1 for p in after if p["latin_ratio"] > 0.3) / max(len(after), 1)
        if mc > 0.5 and sl > 0.3:
            return LayoutClassificationResult(LayoutType.SEQUENTIAL, (mc+sl)/2, True,
                                              boundary_page=boundary, total_pages=total,
                                              detail=f"MK 0-{boundary-1}, SQ {boundary}-{total-1}")

    def _find_transition(self, pd_list):
        for i in range(1, len(pd_list)):
            if pd_list[i-1]["cyrillic_ratio"] > self.seq_t and pd_list[i]["latin_ratio"] > self.seq_t:
                return i
        return None

    def _find_marker(self, pd_list):
        for p in pd_list:
            if p["cyrillic_ratio"] > 0.8 and p["page_index"] < len(pd_list) * 0.3: continue
            for pat in ALB_BOUNDARY:
                if pat.search(p["full_text"]): return p["page_index"]
            if SEP.search(p["full_text"]) and p["latin_ratio"] > 0.3: return p["page_index"]
        return None

In [ ]:
%%writefile /content/vezilka_v2/phase3_extract/document_segmenter.py
"""Split gazette issues into individual items and match MK↔SQ."""
from __future__ import annotations
import logging, re
from dataclasses import dataclass, field
import numpy as np
from config import DEFAULT_CONFIG, VezilkaConfig
logger = logging.getLogger(__name__)
ITEM_BOUNDARY = re.compile(r"^\s*(\d{3,4})\.\s*$", re.MULTILINE)

@dataclass
class GazetteItem:
    item_number: int; mk_text: str = ""; sq_text: str = ""
    is_bilingual: bool = False; laser_doc_sim: float = 0.0
    valid: bool = True; skip_reason: str = ""

@dataclass
class SegmentationResult:
    items: list[GazetteItem] = field(default_factory=list)
    total_items_mk: int = 0; total_items_sq: int = 0; matched_items: int = 0
    unmatched_mk: list[int] = field(default_factory=list)
    unmatched_sq: list[int] = field(default_factory=list)

class DocumentSegmenter:
    def __init__(self, config=None):
        self.cfg = config or DEFAULT_CONFIG
        self._laser_mk = None; self._laser_sq = None

    def segment(self, mk_text: str, sq_text: str) -> SegmentationResult:
        mk_items = self._split(mk_text); sq_items = self._split(sq_text)
        result = SegmentationResult(total_items_mk=len(mk_items), total_items_sq=len(sq_items))
        if not mk_items and not sq_items:
            item = GazetteItem(0, mk_text.strip(), sq_text.strip(),
                               is_bilingual=bool(mk_text.strip() and sq_text.strip()))
            if item.is_bilingual: result.items.append(item); result.matched_items = 1
            return result
        if not mk_items: mk_items = {0: mk_text.strip()}
        if not sq_items: sq_items = {0: sq_text.strip()}
        matched = set(mk_items) & set(sq_items)
        result.unmatched_mk = sorted(set(mk_items) - set(sq_items))
        result.unmatched_sq = sorted(set(sq_items) - set(mk_items))
        for num in sorted(matched):
            result.items.append(GazetteItem(num, mk_items[num], sq_items[num], is_bilingual=True))
        result.matched_items = len(result.items)
        return result

    @staticmethod
    def _split(text):
        if not text: return {}
        matches = list(ITEM_BOUNDARY.finditer(text))
        if not matches: return {}
        items = {}
        for i, m in enumerate(matches):
            num = int(m.group(1)); start = m.end()
            end = matches[i+1].start() if i+1 < len(matches) else len(text)
            body = text[start:end].strip()
            if body: items[num] = body
        return items

In [ ]:
%%writefile /content/vezilka_v2/phase3_extract/extractor_two_column.py
"""Type A: Two-column extractor (MK left, SQ right)."""
from __future__ import annotations
import logging, re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
import pdfplumber
from utils.text_utils import clean_text, cyrillic_ratio, fix_albanian_encoding, fix_hyphenation, latin_ratio, normalise_whitespace
logger = logging.getLogger(__name__)

@dataclass
class ExtractionResult:
    articles: list = field(default_factory=list)
    raw_mk: str = ""; raw_sq: str = ""; source_pdf: str = ""

class TwoColumnExtractor:
    def __init__(self, tolerance=0.08): self.tol = tolerance

    def extract(self, pdf_path: Path) -> ExtractionResult:
        result = ExtractionResult(source_pdf=str(pdf_path))
        all_mk, all_sq = [], []
        try:
            with pdfplumber.open(str(pdf_path)) as pdf:
                for page in pdf.pages:
                    mk, sq = self._split_page(page)
                    all_mk.extend(mk); all_sq.extend(sq)
        except Exception as e:
            logger.error("Two-column fail %s: %s", pdf_path, e); return result
        result.raw_mk = self._join(all_mk, "mk")
        result.raw_sq = self._join(all_sq, "sq")
        return result

    def _split_page(self, page):
        words = page.extract_words(use_text_flow=True) or []
        if not words: return [], []
        w = float(page.width) if page.width else 612.0
        mid, t = w/2, w*self.tol
        left = [x for x in words if float(x.get("x0",0)) < mid-t]
        right = [x for x in words if float(x.get("x0",0)) >= mid+t]
        return self._to_lines(left), self._to_lines(right)

    @staticmethod
    def _to_lines(words):
        if not words: return []
        words = sorted(words, key=lambda w: (round(float(w.get("top",0)),1), float(w.get("x0",0))))
        lines, cur, prev = [], [], None
        for w in words:
            top = round(float(w.get("top",0)),1)
            if prev is not None and abs(top-prev) > 3:
                if cur: lines.append(" ".join(cur))
                cur = []
            cur.append(w.get("text","")); prev = top
        if cur: lines.append(" ".join(cur))
        return lines

    def _join(self, lines, lang):
        text = "\n".join(lines)
        text = fix_hyphenation(text); text = normalise_whitespace(text)
        if lang == "sq": text = fix_albanian_encoding(text)
        return clean_text(text)

In [ ]:
%%writefile /content/vezilka_v2/phase3_extract/extractor_sequential.py
"""Type B: Sequential extractor (full MK then full SQ)."""
from __future__ import annotations
import logging, re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
import pdfplumber
from utils.text_utils import clean_text, cyrillic_ratio, fix_albanian_encoding, fix_hyphenation, latin_ratio, normalise_whitespace
logger = logging.getLogger(__name__)

@dataclass
class ExtractionResult:
    articles: list = field(default_factory=list)
    raw_mk: str = ""; raw_sq: str = ""; source_pdf: str = ""

class SequentialExtractor:
    def extract(self, pdf_path: Path, boundary_page=None, boundary_block=None) -> ExtractionResult:
        result = ExtractionResult(source_pdf=str(pdf_path))
        try:
            with pdfplumber.open(str(pdf_path)) as pdf:
                if boundary_page is not None:
                    mk = "\n".join((pdf.pages[i].extract_text() or "") for i in range(boundary_page))
                    sq = "\n".join((pdf.pages[i].extract_text() or "") for i in range(boundary_page, len(pdf.pages)))
                else:
                    mk_p, sq_p, found = [], [], False
                    for page in pdf.pages:
                        t = page.extract_text() or ""
                        if not found and latin_ratio(t) > 0.5 and cyrillic_ratio(t) < 0.3:
                            found = True
                        (sq_p if found else mk_p).append(t)
                    mk, sq = "\n".join(mk_p), "\n".join(sq_p)
        except Exception as e:
            logger.error("Sequential fail %s: %s", pdf_path, e); return result
        result.raw_mk = self._clean(mk, "mk"); result.raw_sq = self._clean(sq, "sq")
        return result

    @staticmethod
    def _clean(text, lang):
        text = fix_hyphenation(text); text = normalise_whitespace(text)
        if lang == "sq": text = fix_albanian_encoding(text)
        return clean_text(text)

In [ ]:
%%writefile /content/vezilka_v2/phase3_extract/extractor_ocr.py
"""Type C: OCR/Mixed extractor for pre-2019 PDFs."""
from __future__ import annotations
import logging, re
from dataclasses import dataclass, field
from pathlib import Path
from phase3_extract.language_detector import DetectedLanguage, LanguageDetector
from utils.text_utils import clean_text, cyrillic_ratio, fix_albanian_encoding, fix_hyphenation, latin_ratio, normalise_whitespace
logger = logging.getLogger(__name__)

@dataclass
class ExtractionResult:
    articles: list = field(default_factory=list)
    raw_mk: str = ""; raw_sq: str = ""; source_pdf: str = ""

class OCRExtractor:
    def __init__(self, use_ocr=True): self.use_ocr = use_ocr; self._lang = LanguageDetector()

    def extract(self, pdf_path: Path) -> ExtractionResult:
        result = ExtractionResult(source_pdf=str(pdf_path))
        raw = self._fitz(pdf_path)
        if not raw: raw = self._pdfplumber(pdf_path)
        if not raw: return result
        mk_l, sq_l = [], []
        for line in raw.split("\n"):
            line = line.strip()
            if len(line) < 3: continue
            lang = self._lang.detect(line)
            if lang == DetectedLanguage.MACEDONIAN: mk_l.append(line)
            elif lang == DetectedLanguage.ALBANIAN: sq_l.append(line)
            elif cyrillic_ratio(line) > 0.5: mk_l.append(line)
            elif latin_ratio(line) > 0.5: sq_l.append(line)
        result.raw_mk = self._clean("\n".join(mk_l), "mk")
        result.raw_sq = self._clean("\n".join(sq_l), "sq")
        return result

    @staticmethod
    def _fitz(path):
        try:
            import fitz
            doc = fitz.open(str(path)); t = "\n\n".join(p.get_text() for p in doc); doc.close(); return t
        except: return ""

    @staticmethod
    def _pdfplumber(path):
        try:
            import pdfplumber
            with pdfplumber.open(str(path)) as pdf:
                return "\n\n".join((p.extract_text() or "") for p in pdf.pages)
        except: return ""

    @staticmethod
    def _clean(text, lang):
        text = fix_hyphenation(text); text = normalise_whitespace(text)
        if lang == "sq": text = fix_albanian_encoding(text)
        return clean_text(text)

In [ ]:
%%writefile /content/vezilka_v2/phase4_align/__init__.py
"""Phase 4 — Alignment."""

In [ ]:
%%writefile /content/vezilka_v2/phase4_align/structural_aligner.py
"""Strategy 1: Structural alignment via Член N ↔ Neni N matching."""
from __future__ import annotations
import logging, math, re
from dataclasses import dataclass, field
from typing import Optional
from config import DEFAULT_CONFIG, VezilkaConfig
logger = logging.getLogger(__name__)

@dataclass
class AlignedPair:
    mk: str; sq: str; mk_indices: list[int] = field(default_factory=list)
    sq_indices: list[int] = field(default_factory=list)
    alignment_type: str = "1-1"; article_number: Optional[int] = None; dp_cost: float = 0.0

@dataclass
class StructuralAlignmentResult:
    pairs: list[AlignedPair] = field(default_factory=list)
    matched_articles: int = 0
    unmatched_mk_articles: list[int] = field(default_factory=list)
    unmatched_sq_articles: list[int] = field(default_factory=list)
    strategy: str = "structural"

class StructuralAligner:
    def __init__(self, config=None):
        self.cfg = config or DEFAULT_CONFIG
        self._mk_pat = re.compile(self.cfg.structural_article_pattern_mk, re.MULTILINE)
        self._sq_pat = re.compile(self.cfg.structural_article_pattern_sq, re.MULTILINE)

    def align(self, mk_text, sq_text):
        result = StructuralAlignmentResult()
        mk_art = self._articles(mk_text, self._mk_pat)
        sq_art = self._articles(sq_text, self._sq_pat)
        if not mk_art or not sq_art: return result
        matched = sorted(set(mk_art) & set(sq_art))
        result.unmatched_mk_articles = sorted(set(mk_art) - set(sq_art))
        result.unmatched_sq_articles = sorted(set(sq_art) - set(mk_art))
        result.matched_articles = len(matched)
        for num in matched:
            ms = self._split(mk_art[num]); ss = self._split(sq_art[num])
            if ms and ss: result.pairs.extend(self._dp(ms, ss, num))
        logger.info("Structural: %d articles → %d pairs", result.matched_articles, len(result.pairs))
        return result

    def has_articles(self, mk, sq):
        return bool(self._mk_pat.search(mk) and self._sq_pat.search(sq))

    @staticmethod
    def _articles(text, pat):
        matches = list(pat.finditer(text))
        if not matches: return {}
        r = {}
        for i, m in enumerate(matches):
            num = int(m.group(1)); s = m.end(); e = matches[i+1].start() if i+1 < len(matches) else len(text)
            b = text[s:e].strip()
            if len(b) >= 5: r[num] = b
        return r

    @staticmethod
    def _split(text):
        if not text.strip(): return []
        try:
            from sentence_splitter import SentenceSplitter
            return [s.strip() for s in SentenceSplitter(language="en").split(text) if s.strip()]
        except ImportError:
            return [s.strip() for s in re.split(r"(?<=[.!?;])\s+", text) if s.strip()]

    def _dp(self, mk, sq, art_num):
        n, m = len(mk), len(sq); INF = float("inf")
        cost = [[INF]*(m+1) for _ in range(n+1)]; back = [[None]*(m+1) for _ in range(n+1)]
        cost[0][0] = 0.0; mr = self.cfg.gc_mean_char_ratio; v = self.cfg.gc_variance
        def _c(ml, sl):
            if sl == 0: return INF if ml > 0 else 0.0
            r = ml/sl; return (math.log(r/mr)**2)/v if r > 0 else INF
        for i in range(n+1):
            for j in range(m+1):
                if cost[i][j] == INF: continue
                c0 = cost[i][j]
                if i<n and j<m:
                    c = c0 + _c(len(mk[i]), len(sq[j]))
                    if c < cost[i+1][j+1]: cost[i+1][j+1] = c; back[i+1][j+1] = (i,j,"1-1")
                if i<n and j+1<m:
                    c = c0 + _c(len(mk[i]), len(sq[j])+len(sq[j+1]))
                    if c < cost[i+1][j+2]: cost[i+1][j+2] = c; back[i+1][j+2] = (i,j,"1-2")
                if i+1<n and j<m:
                    c = c0 + _c(len(mk[i])+len(mk[i+1]), len(sq[j]))
                    if c < cost[i+2][j+1]: cost[i+2][j+1] = c; back[i+2][j+1] = (i,j,"2-1")
        pairs, ci, cj = [], n, m; raw = []
        while ci > 0 or cj > 0:
            if back[ci][cj] is None: break
            pi, pj, at = back[ci][cj]; raw.append((pi,pj,ci,cj,at)); ci,cj = pi,pj
        for pi,pj,ci,cj,at in reversed(raw):
            if at == "1-1": mt,st,mi,si = mk[pi],sq[pj],[pi],[pj]
            elif at == "1-2": mt,st,mi,si = mk[pi],sq[pj]+" "+sq[pj+1],[pi],[pj,pj+1]
            elif at == "2-1": mt,st,mi,si = mk[pi]+" "+mk[pi+1],sq[pj],[pi,pi+1],[pj]
            else: continue
            pairs.append(AlignedPair(mk=mt,sq=st,mk_indices=mi,sq_indices=si,alignment_type=at,article_number=art_num))
        return pairs

In [ ]:
%%writefile /content/vezilka_v2/phase4_align/gale_church.py
"""Strategy 3: Gale-Church statistical alignment (last resort)."""
from __future__ import annotations
import logging, math, re
from dataclasses import dataclass, field
from config import DEFAULT_CONFIG, VezilkaConfig
logger = logging.getLogger(__name__)
APROB = {"1-1":0.89,"1-0":0.005,"0-1":0.005,"1-2":0.05,"2-1":0.05,"2-2":0.01}

@dataclass
class GCPair:
    mk: str; sq: str; mk_indices: list[int] = field(default_factory=list)
    sq_indices: list[int] = field(default_factory=list)
    alignment_type: str = "1-1"; gc_cost: float = 0.0

@dataclass
class GaleChurchResult:
    pairs: list[GCPair] = field(default_factory=list); strategy: str = "gale_church"

class GaleChurchAligner:
    def __init__(self, config=None):
        self.cfg = config or DEFAULT_CONFIG
        self.mr = self.cfg.gc_mean_char_ratio; self.var = self.cfg.gc_variance

    def align(self, mk_text, sq_text):
        ms = self._split(mk_text); ss = self._split(sq_text)
        if not ms or not ss: return GaleChurchResult()
        pairs = self._dp(ms, ss)
        logger.info("GC: %d MK + %d SQ → %d pairs", len(ms), len(ss), len(pairs))
        return GaleChurchResult(pairs=pairs)

    def _dp(self, mk, sq):
        n, m = len(mk), len(sq); INF = float("inf")
        cost = [[INF]*(m+1) for _ in range(n+1)]; back = [[None]*(m+1) for _ in range(n+1)]
        cost[0][0] = 0.0
        MOVES = [(1,1,"1-1"),(1,0,"1-0"),(0,1,"0-1"),(1,2,"1-2"),(2,1,"2-1"),(2,2,"2-2")]
        for i in range(n+1):
            for j in range(m+1):
                if cost[i][j] == INF: continue
                c0 = cost[i][j]
                for di,dj,at in MOVES:
                    ni,nj = i+di,j+dj
                    if ni > n or nj > m: continue
                    ml = sum(len(mk[i+k]) for k in range(di))
                    sl = sum(len(sq[j+k]) for k in range(dj))
                    c = c0 + self._cost(ml, sl, at)
                    if c < cost[ni][nj]: cost[ni][nj] = c; back[ni][nj] = (i,j,at)
        raw, ci, cj = [], n, m
        while ci > 0 or cj > 0:
            if back[ci][cj] is None: break
            pi,pj,at = back[ci][cj]; raw.append((pi,pj,ci,cj,at)); ci,cj = pi,pj
        pairs = []
        for pi,pj,ci,cj,at in reversed(raw):
            di,dj = ci-pi,cj-pj
            mt = " ".join(mk[pi:ci]) if di > 0 else ""
            st = " ".join(sq[pj:cj]) if dj > 0 else ""
            if mt.strip() and st.strip():
                pairs.append(GCPair(mk=mt, sq=st, mk_indices=list(range(pi,ci)),
                                    sq_indices=list(range(pj,cj)), alignment_type=at))
        return pairs

    def _cost(self, ml, sl, at):
        prior = APROB.get(at, 0.001)
        if sl == 0 and ml == 0: pen = 0
        elif sl == 0: pen = 100
        else:
            r = ml/sl; pen = (math.log(r/self.mr)**2)/self.var if r > 0 else 100
        return -math.log(prior+1e-12) + pen

    @staticmethod
    def _split(text):
        if not text.strip(): return []
        try:
            from sentence_splitter import SentenceSplitter
            return [s.strip() for s in SentenceSplitter(language="en").split(text) if s.strip()]
        except ImportError:
            return [s.strip() for s in re.split(r"(?<=[.!?;])\s+", text) if s.strip()]

In [ ]:
%%writefile /content/vezilka_v2/phase4_align/dense_retrieval_aligner.py
"""Strategy 2: Dense retrieval via LASER3 mutual-NN + monotonicity."""
from __future__ import annotations
import logging, re
from dataclasses import dataclass, field
import numpy as np
from config import DEFAULT_CONFIG, VezilkaConfig
logger = logging.getLogger(__name__)

@dataclass
class DenseAlignedPair:
    mk: str; sq: str; mk_index: int = 0; sq_index: int = 0
    similarity: float = 0.0; alignment_type: str = "dense_retrieval"

@dataclass
class DenseRetrievalResult:
    pairs: list[DenseAlignedPair] = field(default_factory=list)
    total_mk_sentences: int = 0; total_sq_sentences: int = 0
    mutual_nn_before_monotone: int = 0; strategy: str = "dense_retrieval"

class DenseRetrievalAligner:
    def __init__(self, config=None):
        self.cfg = config or DEFAULT_CONFIG
        self._mk_enc = None; self._sq_enc = None

    def align(self, mk_text, sq_text):
        result = DenseRetrievalResult()
        ms = self._split(mk_text); ss = self._split(sq_text)
        result.total_mk_sentences = len(ms); result.total_sq_sentences = len(ss)
        if len(ms) < 2 or len(ss) < 2: return result
        mke, sqe = self._get_enc()
        if mke is None: return result
        try:
            me = np.asarray(mke.encode_sentences(ms), dtype=np.float32)
            se = np.asarray(sqe.encode_sentences(ss), dtype=np.float32)
        except Exception as e:
            logger.error("LASER3 encode fail: %s", e); return result
        me /= (np.linalg.norm(me, axis=1, keepdims=True)+1e-12)
        se /= (np.linalg.norm(se, axis=1, keepdims=True)+1e-12)
        fi, fs = self._nn(me, se); bi, bs = self._nn(se, me)
        mn = self.cfg.dense_retrieval_min_similarity; cands = []
        for i in range(len(ms)):
            j = int(fi[i])
            if int(bi[j]) == i and float(fs[i]) >= mn:
                cands.append((i, j, float(fs[i])))
        result.mutual_nn_before_monotone = len(cands)
        if self.cfg.dense_retrieval_monotonicity and cands:
            cands = self._monotone(cands)
        for i, j, sim in cands:
            result.pairs.append(DenseAlignedPair(mk=ms[i], sq=ss[j], mk_index=i, sq_index=j, similarity=sim))
        logger.info("Dense: %d MK + %d SQ → %d mutual NN → %d monotone", len(ms), len(ss), result.mutual_nn_before_monotone, len(result.pairs))
        return result

    def _nn(self, q, t):
        if self.cfg.dense_retrieval_use_faiss:
            try:
                import faiss
                idx = faiss.IndexFlatIP(t.shape[1]); idx.add(t)
                s, i = idx.search(q, 1); return i.flatten(), s.flatten()
            except ImportError: pass
        sim = q @ t.T; i = np.argmax(sim, axis=1); s = sim[np.arange(len(q)), i]
        return i, s

    @staticmethod
    def _monotone(cands):
        cands.sort(key=lambda t: (t[0], t[1])); n = len(cands)
        dp = [0.0]*n; prev = [-1]*n
        for k in range(n):
            dp[k] = cands[k][2]; prev[k] = -1
            for p in range(k):
                if cands[p][1] < cands[k][1]:
                    sc = dp[p] + cands[k][2]
                    if sc > dp[k]: dp[k] = sc; prev[k] = p
        best = int(np.argmax(dp)); chain = []
        idx = best
        while idx >= 0: chain.append(idx); idx = prev[idx]
        chain.reverse(); return [cands[k] for k in chain]

    def _get_enc(self):
        if self._mk_enc is None:
            try:
                from laser_encoders import LaserEncoderPipeline
                self._mk_enc = LaserEncoderPipeline(lang="mkd")
                self._sq_enc = LaserEncoderPipeline(lang="sqi")
            except Exception as e:
                logger.error("LASER3 init fail: %s", e); self._mk_enc = False; self._sq_enc = False
        if self._mk_enc is False: return None, None
        return self._mk_enc, self._sq_enc

    @staticmethod
    def _split(text):
        if not text.strip(): return []
        try:
            from sentence_splitter import SentenceSplitter
            return [s.strip() for s in SentenceSplitter(language="en").split(text) if s.strip()]
        except ImportError:
            return [s.strip() for s in re.split(r"(?<=[.!?;])\s+", text) if s.strip()]

In [ ]:
%%writefile /content/vezilka_v2/phase4_align/translation_scorer.py
"""MarianMT forward translation + chrF++ scoring."""
from __future__ import annotations
import logging
from config import DEFAULT_CONFIG, VezilkaConfig
logger = logging.getLogger(__name__)

class TranslationScorer:
    def __init__(self, config=None):
        self.cfg = config or DEFAULT_CONFIG
        self._translator = None; self._chrf = None

    def score_pairs(self, mk_sentences, sq_sentences, batch_size=None):
        if not mk_sentences: return []
        bs = batch_size or self.cfg.translation_batch_size
        tr = self._get_tr(); ch = self._get_ch()
        if tr is None or ch is None: return [0.0]*len(mk_sentences)
        scores = []
        for s in range(0, len(mk_sentences), bs):
            bm = mk_sentences[s:s+bs]; bq = sq_sentences[s:s+bs]
            try:
                out = tr(bm, max_length=512, batch_size=len(bm))
                mt = [o["translation_text"] for o in out]
            except: scores.extend([0.0]*len(bm)); continue
            for m, r in zip(mt, bq):
                try: scores.append(ch.sentence_score(m, [r]).score / 100.0)
                except: scores.append(0.0)
        return scores

    def _get_tr(self):
        if self._translator is None:
            try:
                from transformers import pipeline as hp
                self._translator = hp("translation", model=self.cfg.bt_mk_to_sq_model, device=-1)
            except: self._translator = False
        return self._translator if self._translator is not False else None

    def _get_ch(self):
        if self._chrf is None:
            try:
                from sacrebleu.metrics import CHRF
                self._chrf = CHRF(word_order=2)
            except: self._chrf = False
        return self._chrf if self._chrf is not False else None

In [ ]:
%%writefile /content/vezilka_v2/phase4_align/aligner_orchestrator.py
"""Three-strategy alignment cascade per gazette item."""
from __future__ import annotations
import logging
from dataclasses import dataclass, field
from typing import Optional
from config import DEFAULT_CONFIG, VezilkaConfig
from phase4_align.dense_retrieval_aligner import DenseRetrievalAligner
from phase4_align.gale_church import GaleChurchAligner
from phase4_align.structural_aligner import StructuralAligner
logger = logging.getLogger(__name__)

@dataclass
class CandidatePair:
    mk: str; sq: str; pdf_id: str = ""; item_number: int = 0
    article_number: Optional[int] = None; alignment_strategy: str = ""
    layout_type: str = ""; mk_word_count: int = 0; sq_word_count: int = 0
    labse_score: float = 0.0; laser3_score: float = 0.0; comet_qe_score: float = 0.0
    back_translation_score: float = 0.0; length_ratio_score: float = 0.0
    blended_confidence: float = 0.0; tier_reached: int = 0
    is_structural: bool = False; rejection_reason: str = ""

@dataclass
class OrchestrationResult:
    candidates: list[CandidatePair] = field(default_factory=list)
    strategy_used: str = ""; structural_articles: int = 0
    dense_pairs: int = 0; gc_pairs: int = 0

class AlignerOrchestrator:
    def __init__(self, config=None):
        self.cfg = config or DEFAULT_CONFIG
        self.structural = StructuralAligner(self.cfg)
        self.dense = DenseRetrievalAligner(self.cfg)
        self.gc = GaleChurchAligner(self.cfg)

    def align_item(self, mk_text, sq_text, pdf_id="", item_number=0, layout_type=""):
        result = OrchestrationResult()
        if not mk_text.strip() or not sq_text.strip(): return result
        if self.structural.has_articles(mk_text, sq_text):
            s1 = self.structural.align(mk_text, sq_text)
            if s1.pairs:
                result.strategy_used = "structural"; result.structural_articles = s1.matched_articles
                for p in s1.pairs:
                    result.candidates.append(CandidatePair(mk=p.mk, sq=p.sq, pdf_id=pdf_id,
                        item_number=item_number, article_number=p.article_number,
                        alignment_strategy="structural", layout_type=layout_type,
                        mk_word_count=len(p.mk.split()), sq_word_count=len(p.sq.split()), is_structural=True))
                return result
        s2 = self.dense.align(mk_text, sq_text)
        if len(s2.pairs) >= self.cfg.dense_retrieval_min_pairs_fallback:
            result.strategy_used = "dense_retrieval"; result.dense_pairs = len(s2.pairs)
            for p in s2.pairs:
                result.candidates.append(CandidatePair(mk=p.mk, sq=p.sq, pdf_id=pdf_id,
                    item_number=item_number, alignment_strategy="dense_retrieval",
                    layout_type=layout_type, mk_word_count=len(p.mk.split()),
                    sq_word_count=len(p.sq.split()), laser3_score=p.similarity))
            return result
        s3 = self.gc.align(mk_text, sq_text)
        result.strategy_used = "gale_church"; result.gc_pairs = len(s3.pairs)
        for p in s3.pairs:
            result.candidates.append(CandidatePair(mk=p.mk, sq=p.sq, pdf_id=pdf_id,
                item_number=item_number, alignment_strategy="gale_church",
                layout_type=layout_type, mk_word_count=len(p.mk.split()),
                sq_word_count=len(p.sq.split())))
        return result

In [ ]:
%%writefile /content/vezilka_v2/phase5_clean/__init__.py
"""Phase 5 — Cleaning."""

In [ ]:
%%writefile /content/vezilka_v2/phase5_clean/semantic_validator.py
"""Tiered semantic validation: LaBSE + LASER3 + COMET-QE + back-translation."""
from __future__ import annotations
import json, logging
from pathlib import Path
import numpy as np
from config import DEFAULT_CONFIG, VezilkaConfig
from phase4_align.aligner_orchestrator import CandidatePair
logger = logging.getLogger(__name__)

class SemanticValidator:
    def __init__(self, config=None):
        self.cfg = config or DEFAULT_CONFIG
        self._labse = None; self._laser_mk = None; self._laser_sq = None
        self._comet = None; self._fwd = None; self._bwd = None; self._chrf = None

    def validate(self, pairs):
        if not pairs: return pairs
        mk = [p.mk for p in pairs]; sq = [p.sq for p in pairs]
        # Tier 1
        logger.info("Tier 1: length + LaBSE + LASER3 on %d pairs", len(pairs))
        ls = self._len_ratio(mk, sq); lb = self._labse(mk, sq); la = self._laser3(mk, sq)
        for i, p in enumerate(pairs):
            p.length_ratio_score = ls[i]; p.labse_score = lb[i]; p.laser3_score = la[i]; p.tier_reached = 1
        for p in pairs:
            r = self._hard1(p)
            if r: p.rejection_reason = r; p.blended_confidence = 0.0
        surv = [p for p in pairs if not p.rejection_reason]
        logger.info("Tier 1: %d/%d survive", len(surv), len(pairs))
        # Tier 2
        if self.cfg.run_comet_qe:
            t2 = [p for p in surv if not p.is_structural]
            if t2:
                logger.info("Tier 2: COMET-QE on %d pairs", len(t2))
                cs = self._comet_qe([p.mk for p in t2], [p.sq for p in t2])
                for p, s in zip(t2, cs):
                    p.comet_qe_score = s; p.tier_reached = 2
                    if s < self.cfg.hard_reject_comet_qe: p.rejection_reason = f"hard_comet_{s:.3f}"
        surv = [p for p in pairs if not p.rejection_reason]
        # Tier 3
        if self.cfg.run_back_translation:
            t3 = [p for p in surv if not p.is_structural and p.labse_score > 0.75 and 0 < p.comet_qe_score < 0.75]
            if t3:
                logger.info("Tier 3: back-translation on %d pairs", len(t3))
                bs = self._backtrans([p.mk for p in t3], [p.sq for p in t3])
                for p, s in zip(t3, bs): p.back_translation_score = s; p.tier_reached = 3
        for p in pairs:
            if p.rejection_reason: continue
            p.blended_confidence = self._blend(p)
            if p.blended_confidence < self.cfg.blended_min_score:
                p.rejection_reason = f"low_blend_{p.blended_confidence:.3f}"
        acc = [p for p in pairs if not p.rejection_reason]
        rej = [p for p in pairs if p.rejection_reason]
        logger.info("Final: %d accepted, %d rejected", len(acc), len(rej))
        self._log_rej(rej); return pairs

    def _len_ratio(self, mk, sq):
        scores = []
        for m, s in zip(mk, sq):
            mw, sw = len(m.split()), len(s.split())
            if sw == 0 or mw == 0: scores.append(0.0); continue
            r = mw / sw
            if self.cfg.min_length_ratio <= r <= self.cfg.max_length_ratio:
                scores.append(max(0.0, 1.0 - abs(r-1.0)/(self.cfg.max_length_ratio-1.0)))
            else: scores.append(0.0)
        return scores

    def _labse(self, mk, sq):
        m = self._get_labse()
        if m is None: return [0.0]*len(mk)
        try:
            me = m.encode(mk, batch_size=self.cfg.labse_batch_size, show_progress_bar=False, normalize_embeddings=True)
            se = m.encode(sq, batch_size=self.cfg.labse_batch_size, show_progress_bar=False, normalize_embeddings=True)
            return [float(s) for s in np.sum(me*se, axis=1)]
        except Exception as e:
            logger.error("LaBSE fail: %s", e); return [0.0]*len(mk)

    def _laser3(self, mk, sq):
        me, se = self._get_laser()
        if me is None: return [0.0]*len(mk)
        try:
            m = np.asarray(me.encode_sentences(mk), dtype=np.float32)
            s = np.asarray(se.encode_sentences(sq), dtype=np.float32)
            m /= (np.linalg.norm(m, axis=1, keepdims=True)+1e-12)
            s /= (np.linalg.norm(s, axis=1, keepdims=True)+1e-12)
            return [float(v) for v in np.sum(m*s, axis=1)]
        except Exception as e:
            logger.error("LASER3 fail: %s", e); return [0.0]*len(mk)

    def _comet_qe(self, mk, sq):
        m = self._get_comet()
        if m is None: return [0.0]*len(mk)
        try:
            data = [{"src": a, "mt": b} for a, b in zip(mk, sq)]
            out = m.predict(data, batch_size=16, gpus=1 if self.cfg.device=="cuda" else 0)
            return [float(s) for s in out.scores]
        except Exception as e:
            logger.error("COMET fail: %s", e); return [0.0]*len(mk)

    def _backtrans(self, mk, sq):
        f, b, c = self._get_bt()
        if f is None: return [0.0]*len(mk)
        bs = self.cfg.bt_batch_size; scores = []
        for i in range(0, len(mk), bs):
            bm, bq = mk[i:i+bs], sq[i:i+bs]
            try:
                mt = [o["translation_text"] for o in f(bm, max_length=512, batch_size=len(bm))]
                bt = [o["translation_text"] for o in b(mt, max_length=512, batch_size=len(mt))]
                for mo, sc, ms, bm_ in zip(bm, bq, mt, bt):
                    cf = c.sentence_score(ms, [sc]).score / 100.0
                    cb = c.sentence_score(bm_, [mo]).score / 100.0
                    scores.append(2*cf*cb/(cf+cb) if cf+cb > 0 else 0.0)
            except: scores.extend([0.0]*len(bm))
        return scores

    def _blend(self, p):
        if p.is_structural:
            return self.cfg.w_structural_labse*p.labse_score + self.cfg.w_structural_laser3*p.laser3_score + self.cfg.w_structural_length*p.length_ratio_score
        return (self.cfg.w_nonstructural_labse*p.labse_score + self.cfg.w_nonstructural_laser3*p.laser3_score +
                self.cfg.w_nonstructural_comet_qe*p.comet_qe_score + self.cfg.w_nonstructural_backtranslation*p.back_translation_score +
                self.cfg.w_nonstructural_length*p.length_ratio_score)

    def _hard1(self, p):
        if p.labse_score < self.cfg.hard_reject_labse: return f"labse_{p.labse_score:.3f}"
        if p.laser3_score < self.cfg.hard_reject_laser3: return f"laser3_{p.laser3_score:.3f}"
        if p.length_ratio_score < self.cfg.hard_reject_length_ratio: return f"length_{p.length_ratio_score:.3f}"
        return ""

    def _log_rej(self, rej):
        if not rej: return
        p = self.cfg.rejected_pairs_log; p.parent.mkdir(parents=True, exist_ok=True)
        try:
            with open(p, "a") as f:
                for r in rej:
                    f.write(json.dumps({"mk": r.mk[:200], "sq": r.sq[:200], "reason": r.rejection_reason,
                        "labse": round(r.labse_score,4), "laser3": round(r.laser3_score,4)}, ensure_ascii=False)+"\n")
        except: pass

    def _get_labse(self):
        if self._labse is None:
            try:
                from sentence_transformers import SentenceTransformer
                self._labse = SentenceTransformer(self.cfg.labse_model, device=self.cfg.device)
            except: self._labse = False
        return self._labse if self._labse is not False else None

    def _get_laser(self):
        if self._laser_mk is None:
            try:
                from laser_encoders import LaserEncoderPipeline
                self._laser_mk = LaserEncoderPipeline(lang="mkd")
                self._laser_sq = LaserEncoderPipeline(lang="sqi")
            except: self._laser_mk = False; self._laser_sq = False
        if self._laser_mk is False: return None, None
        return self._laser_mk, self._laser_sq

    def _get_comet(self):
        if self._comet is None:
            try:
                from comet import download_model, load_from_checkpoint
                self._comet = load_from_checkpoint(download_model(self.cfg.comet_qe_model))
            except: self._comet = False
        return self._comet if self._comet is not False else None

    def _get_bt(self):
        if self._fwd is None:
            try:
                from transformers import pipeline as hp
                from sacrebleu.metrics import CHRF
                self._fwd = hp("translation", model=self.cfg.bt_mk_to_sq_model, device=0 if self.cfg.device=="cuda" else -1)
                self._bwd = hp("translation", model=self.cfg.bt_sq_to_mk_model, device=0 if self.cfg.device=="cuda" else -1)
                self._chrf = CHRF(word_order=2)
            except: self._fwd = False; self._bwd = False; self._chrf = False
        if self._fwd is False: return None, None, None
        return self._fwd, self._bwd, self._chrf

In [ ]:
%%writefile /content/vezilka_v2/phase5_clean/noise_filter.py
"""Pre-alignment noise removal."""
from __future__ import annotations
import logging
from config import DEFAULT_CONFIG
from utils.text_utils import cyrillic_ratio, digit_fraction, is_noise_line, latin_ratio, strip_headers_footers
logger = logging.getLogger(__name__)

class NoiseFilter:
    def __init__(self, config=None): self.cfg = config or DEFAULT_CONFIG

    def clean_mk(self, text):
        text = strip_headers_footers(text)
        if cyrillic_ratio(text) < self.cfg.mk_min_cyrillic and len(text.strip()) > 20: return ""
        return text.strip()

    def clean_sq(self, text):
        text = strip_headers_footers(text)
        if latin_ratio(text) < self.cfg.sq_min_latin and len(text.strip()) > 20: return ""
        return text.strip()

In [ ]:
%%writefile /content/vezilka_v2/phase5_clean/pair_filter.py
"""Post-validation heuristic pair filter."""
from __future__ import annotations
import logging
from config import DEFAULT_CONFIG
from phase4_align.aligner_orchestrator import CandidatePair
from utils.text_utils import cyrillic_ratio, digit_fraction, latin_ratio, word_count
logger = logging.getLogger(__name__)

class PairFilter:
    def __init__(self, config=None): self.cfg = config or DEFAULT_CONFIG

    def filter(self, pairs):
        kept = []
        for p in pairs:
            if p.rejection_reason: continue
            mw, sw = word_count(p.mk), word_count(p.sq)
            if mw < self.cfg.min_words or sw < self.cfg.min_words: continue
            if mw > self.cfg.max_words or sw > self.cfg.max_words: continue
            r = mw / max(sw, 1)
            if r < self.cfg.min_length_ratio or r > self.cfg.max_length_ratio: continue
            if digit_fraction(p.mk) > self.cfg.max_digit_fraction: continue
            if digit_fraction(p.sq) > self.cfg.max_digit_fraction: continue
            kept.append(p)
        logger.info("PairFilter: %d → %d", len(pairs), len(kept))
        return kept

In [ ]:
%%writefile /content/vezilka_v2/phase5_clean/deduplicator.py
"""MinHash LSH near-duplicate removal."""
from __future__ import annotations
import logging
from config import DEFAULT_CONFIG
logger = logging.getLogger(__name__)

class Deduplicator:
    def __init__(self, config=None): self.cfg = config or DEFAULT_CONFIG

    def deduplicate(self, pairs):
        if not pairs: return pairs
        try: from datasketch import MinHash, MinHashLSH
        except: logger.warning("datasketch missing"); return pairs
        t = self.cfg.minhash_threshold; np_ = self.cfg.minhash_num_perm
        mk_lsh = MinHashLSH(threshold=t, num_perm=np_)
        keep = []
        for i, p in enumerate(pairs):
            mh = MinHash(num_perm=np_)
            for j in range(len(p.mk)-2): mh.update(p.mk[j:j+3].lower().encode())
            try:
                if mk_lsh.query(mh): continue
            except: pass
            try: mk_lsh.insert(f"mk_{i}", mh)
            except: pass
            keep.append(p)
        logger.info("Dedup: %d → %d", len(pairs), len(keep))
        return keep

In [ ]:
%%writefile /content/vezilka_v2/phase5_clean/exporter.py
"""Export final corpus: TSV + JSONL + splits."""
from __future__ import annotations
import json, logging, random
from pathlib import Path
import pandas as pd
from config import DEFAULT_CONFIG
from phase4_align.aligner_orchestrator import CandidatePair
logger = logging.getLogger(__name__)

class Exporter:
    def __init__(self, config=None): self.cfg = config or DEFAULT_CONFIG

    def export(self, pairs, seed=42):
        self.cfg.output_dir.mkdir(parents=True, exist_ok=True)
        records = [self._to_dict(p) for p in pairs]
        if not records: return {}
        exported = {}
        if "tsv" in self.cfg.export_formats:
            p = self.cfg.output_dir / "vezilka_v2_corpus.tsv"
            pd.DataFrame(records).to_csv(p, sep="\t", index=False); exported["tsv"] = p
        if "jsonl" in self.cfg.export_formats:
            p = self.cfg.output_dir / "vezilka_v2_corpus.jsonl"
            with open(p, "w") as f:
                for r in records: f.write(json.dumps(r, ensure_ascii=False)+"\n")
            exported["jsonl"] = p
        random.seed(seed); random.shuffle(records); n = len(records)
        nt = int(n * self.cfg.train_fraction); nv = int(n * self.cfg.val_fraction)
        for name, data in [("train", records[:nt]), ("val", records[nt:nt+nv]), ("test", records[nt+nv:])]:
            if "tsv" in self.cfg.export_formats:
                pd.DataFrame(data).to_csv(self.cfg.output_dir / f"vezilka_v2_{name}.tsv", sep="\t", index=False)
            if "jsonl" in self.cfg.export_formats:
                with open(self.cfg.output_dir / f"vezilka_v2_{name}.jsonl", "w") as f:
                    for r in data: f.write(json.dumps(r, ensure_ascii=False)+"\n")
        logger.info("Exported %d pairs: train=%d val=%d test=%d", n, nt, nv, n-nt-nv)
        return exported

    @staticmethod
    def _to_dict(p):
        return {"mk": p.mk, "sq": p.sq, "pdf_id": p.pdf_id, "item_number": p.item_number,
                "article_number": p.article_number or 0, "alignment_strategy": p.alignment_strategy,
                "layout_type": p.layout_type, "labse_score": round(p.labse_score,4),
                "laser3_score": round(p.laser3_score,4), "comet_qe_score": round(p.comet_qe_score,4),
                "back_translation_score": round(p.back_translation_score,4),
                "length_ratio_score": round(p.length_ratio_score,4),
                "blended_confidence": round(p.blended_confidence,4),
                "mk_word_count": p.mk_word_count, "sq_word_count": p.sq_word_count, "tier_reached": p.tier_reached}

In [ ]:
%%writefile /content/vezilka_v2/pipeline.py
"""Master pipeline: extract → segment → align → validate → export."""
from __future__ import annotations
import json, logging, sys, time
from pathlib import Path
from tqdm.auto import tqdm
from config import DEFAULT_CONFIG, VezilkaConfig
from phase3_extract.document_segmenter import DocumentSegmenter
from phase3_extract.extractor_ocr import OCRExtractor
from phase3_extract.extractor_sequential import SequentialExtractor
from phase3_extract.extractor_two_column import TwoColumnExtractor
from phase3_extract.gatekeeper import Gatekeeper, SkipReason
from phase3_extract.layout_classifier import LayoutClassifier, LayoutType
from phase4_align.aligner_orchestrator import AlignerOrchestrator, CandidatePair
from phase5_clean.deduplicator import Deduplicator
from phase5_clean.exporter import Exporter
from phase5_clean.noise_filter import NoiseFilter
from phase5_clean.pair_filter import PairFilter
from phase5_clean.semantic_validator import SemanticValidator
from utils.logging_config import setup_logging
logger = logging.getLogger(__name__)

class VezilkaPipeline:
    def __init__(self, config=None):
        self.cfg = config or DEFAULT_CONFIG; self.cfg.ensure_dirs()
        self.gatekeeper = Gatekeeper(self.cfg)
        self.classifier = LayoutClassifier(
            column_split_tolerance=self.cfg.column_split_tolerance,
            two_column_min_fraction=self.cfg.two_column_min_page_fraction,
            cyrillic_threshold=self.cfg.cyrillic_threshold,
            latin_threshold=self.cfg.latin_threshold,
            sequential_transition_threshold=self.cfg.sequential_transition_threshold)
        self.segmenter = DocumentSegmenter(self.cfg)
        self.ext_two = TwoColumnExtractor(tolerance=self.cfg.column_split_tolerance)
        self.ext_seq = SequentialExtractor()
        self.ext_ocr = OCRExtractor(use_ocr=True)
        self.noise = NoiseFilter(self.cfg)
        self.orch = AlignerOrchestrator(self.cfg)
        self.validator = SemanticValidator(self.cfg)
        self.pair_filter = PairFilter(self.cfg)
        self.dedup = Deduplicator(self.cfg)
        self.exporter = Exporter(self.cfg)

    def run(self, limit=None, skip_validation=False):
        setup_logging(level=self.cfg.log_level); start = time.time()
        pdfs = sorted(self.cfg.pdf_dir.rglob("*.pdf"))
        if limit: pdfs = pdfs[:limit]
        logger.info("Processing %d PDFs", len(pdfs))
        all_cands = []; stats = {"ok": 0, "skip": 0, "fail": 0, "pairs": 0}
        for pdf_path in tqdm(pdfs, desc="PDFs"):
            pid = self._pid(pdf_path)
            ap = self.cfg.aligned_dir / f"{pid}.jsonl"
            if ap.exists():
                all_cands.extend(self._load(ap)); stats["ok"] += 1; continue
            dec = self.gatekeeper.check(pdf_path)
            if not dec.should_process:
                self.gatekeeper.log_skip(pdf_path, dec); stats["skip"] += 1; continue
            try: layout = self.classifier.classify(pdf_path)
            except Exception as e: self._fail(pdf_path, "classify", e); stats["fail"] += 1; continue
            try: mk, sq = self._extract(pdf_path, layout)
            except Exception as e: self._fail(pdf_path, "extract", e); stats["fail"] += 1; continue
            if not mk.strip() or not sq.strip(): stats["skip"] += 1; continue
            mk = self.noise.clean_mk(mk); sq = self.noise.clean_sq(sq)
            if not mk or not sq: stats["skip"] += 1; continue
            try: seg = self.segmenter.segment(mk, sq)
            except Exception as e: self._fail(pdf_path, "segment", e); stats["fail"] += 1; continue
            item_pairs = []
            for item in seg.items:
                if not item.valid or not item.is_bilingual: continue
                r = self.orch.align_item(item.mk_text, item.sq_text, pid, item.item_number, layout.layout_type.value)
                item_pairs.extend(r.candidates)
            self._save(pid, item_pairs)
            all_cands.extend(item_pairs); stats["ok"] += 1; stats["pairs"] += len(item_pairs)
        logger.info("Extract done: %d ok, %d skip, %d fail, %d raw pairs", stats["ok"], stats["skip"], stats["fail"], stats["pairs"])
        if not skip_validation and all_cands:
            logger.info("Validating %d candidates...", len(all_cands))
            all_cands = self.validator.validate(all_cands)
        accepted = [p for p in all_cands if not p.rejection_reason]
        accepted = self.pair_filter.filter(accepted)
        accepted = self.dedup.deduplicate(accepted)
        exported = self.exporter.export(accepted)
        elapsed = time.time() - start
        logger.info("✅ Done in %.1f min: %d final pairs from %d PDFs", elapsed/60, len(accepted), stats["ok"])
        for f, p in exported.items(): logger.info("  %s → %s", f, p)

    def _extract(self, path, layout):
        lt = layout.layout_type
        if lt == LayoutType.TWO_COLUMN: r = self.ext_two.extract(path)
        elif lt == LayoutType.SEQUENTIAL: r = self.ext_seq.extract(path, layout.boundary_page, layout.boundary_block)
        else: r = self.ext_ocr.extract(path)
        return r.raw_mk, r.raw_sq

    @staticmethod
    def _pid(path):
        parts = path.parts
        try: i = next(j for j, p in enumerate(parts) if p == "pdfs")
        except StopIteration: return path.stem
        return "_".join(parts[i+1:]).replace(".pdf", "")

    def _save(self, pid, pairs):
        p = self.cfg.aligned_dir / f"{pid}.jsonl"; p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w") as f:
            for c in pairs:
                f.write(json.dumps({"mk": c.mk, "sq": c.sq, "pdf_id": c.pdf_id,
                    "item_number": c.item_number, "article_number": c.article_number,
                    "alignment_strategy": c.alignment_strategy, "layout_type": c.layout_type,
                    "mk_word_count": c.mk_word_count, "sq_word_count": c.sq_word_count,
                    "is_structural": c.is_structural}, ensure_ascii=False)+"\n")

    def _load(self, path):
        pairs = []
        with open(path) as f:
            for line in f:
                if not line.strip(): continue
                d = json.loads(line)
                pairs.append(CandidatePair(mk=d["mk"], sq=d["sq"], pdf_id=d.get("pdf_id",""),
                    item_number=d.get("item_number",0), article_number=d.get("article_number"),
                    alignment_strategy=d.get("alignment_strategy",""), layout_type=d.get("layout_type",""),
                    mk_word_count=d.get("mk_word_count",0), sq_word_count=d.get("sq_word_count",0),
                    is_structural=d.get("is_structural",False)))
        return pairs

    def _fail(self, path, phase, err):
        logger.error("FAIL [%s] %s: %s", phase, path, err)
        p = self.cfg.failed_log; p.parent.mkdir(parents=True, exist_ok=True)
        try:
            with open(p, "a") as f:
                f.write(json.dumps({"pdf": str(path), "phase": phase, "error": str(err)}, ensure_ascii=False)+"\n")
        except: pass

## Step 4 — Verify All Modules Compile

In [ ]:
import py_compile, os
base = "/content/vezilka_v2"
files = []
for root, dirs, fnames in os.walk(base):
    for fn in fnames:
        if fn.endswith(".py"):
            files.append(os.path.join(root, fn))

ok, fail = 0, 0
for f in sorted(files):
    try:
        py_compile.compile(f, doraise=True)
        ok += 1
    except py_compile.PyCompileError as e:
        print(f"❌ {f}: {e}")
        fail += 1

print(f"\n{'✅' if fail == 0 else '❌'} {ok} passed, {fail} failed")

In [ ]:
import sys
sys.path.insert(0, "/content/vezilka_v2")

from pipeline import VezilkaPipeline
print("✅ Pipeline imports OK")

from config import DEFAULT_CONFIG
print(f"PDF dir: {DEFAULT_CONFIG.pdf_dir}")
print(f"PDFs found: {len(list(DEFAULT_CONFIG.pdf_dir.rglob('*.pdf')))}")

## Step 5 — Run the Pipeline! 🚀

**Option A**: Quick test (10 PDFs, no validation) — ~2 min  
**Option B**: Test with validation (10 PDFs) — ~10 min  
**Option C**: Full run (all PDFs) — ~1.5–2 hours on T4

In [ ]:
# === CHOOSE YOUR RUN MODE ===
# Option A: Quick test
# LIMIT = 10; SKIP_VAL = True

# Option B: Test with validation
# LIMIT = 10; SKIP_VAL = False

# Option C: Full run
LIMIT = None; SKIP_VAL = False
# ============================

import sys
sys.path.insert(0, "/content/vezilka_v2")

from config import VezilkaConfig
from pipeline import VezilkaPipeline

config = VezilkaConfig(device="cuda" if __import__('torch').cuda.is_available() else "cpu")
pipeline = VezilkaPipeline(config=config)
pipeline.run(limit=LIMIT, skip_validation=SKIP_VAL)

## Step 6 — Inspect Results

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path("/content/vezilka_v2/data/output")
tsv = output_dir / "vezilka_v2_corpus.tsv"

if tsv.exists():
    df = pd.read_csv(tsv, sep="\t")
    print(f"Total pairs: {len(df):,}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nAlignment strategy distribution:")
    print(df['alignment_strategy'].value_counts())
    print(f"\nLayout type distribution:")
    print(df['layout_type'].value_counts())
    print(f"\nBlended confidence stats:")
    print(df['blended_confidence'].describe())
    print(f"\nTier reached distribution:")
    print(df['tier_reached'].value_counts().sort_index())
    print(f"\n--- Sample pairs ---")
    for _, row in df.sample(min(5, len(df))).iterrows():
        print(f"\nMK: {row['mk'][:100]}...")
        print(f"SQ: {row['sq'][:100]}...")
        print(f"   strategy={row['alignment_strategy']}, blended={row['blended_confidence']:.3f}")
else:
    print("No output TSV found. Check logs above for errors.")

# Check logs
for log_name in ["skipped_pdfs.jsonl", "failed_pdfs.jsonl", "rejected_pairs.jsonl"]:
    log_path = Path(f"/content/vezilka_v2/data/{log_name}")
    if log_path.exists():
        count = sum(1 for _ in open(log_path))
        print(f"\n{log_name}: {count:,} entries")

## Step 7 — Upload Results to GCS

In [ ]:
from google.cloud import storage
from pathlib import Path

client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

output_dir = Path("/content/vezilka_v2/data/output")
data_dir = Path("/content/vezilka_v2/data")

# Upload output files
uploaded = 0
for f in output_dir.rglob("*"):
    if f.is_file():
        blob_name = f"{GCS_OUTPUT_PREFIX}{f.relative_to(data_dir)}"
        blob = bucket.blob(blob_name)
        blob.upload_from_filename(str(f))
        uploaded += 1
        print(f"  ↑ gs://{GCS_BUCKET}/{blob_name}")

# Upload logs
for log_name in ["skipped_pdfs.jsonl", "failed_pdfs.jsonl", "rejected_pairs.jsonl"]:
    log_path = data_dir / log_name
    if log_path.exists():
        blob = bucket.blob(f"{GCS_OUTPUT_PREFIX}logs/{log_name}")
        blob.upload_from_filename(str(log_path))
        uploaded += 1
        print(f"  ↑ gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}logs/{log_name}")

print(f"\n✅ Uploaded {uploaded} files to gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}")

## Step 8 — Download Results Locally (optional)

Click the folder icon in the left sidebar → navigate to `/content/vezilka_v2/data/output/` → right-click → Download.

Or use `gsutil` from your local machine:
```bash
gsutil -m cp -r gs://YOUR_BUCKET/output/ ./results/
```